In [6]:
# 06 Sanity Check - Ayush's Modules
# ==================================

import sys
import os

# Add parent directory to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

# Check if path is correct
print("Current directory:", os.getcwd())
print("Parent directory:", os.path.dirname(os.getcwd()))

# List files in eeg_pipeline folder
eeg_pipeline_path = os.path.join(os.path.dirname(os.getcwd()), "eeg_pipeline")
print(f"\nFiles in eeg_pipeline folder:")
if os.path.exists(eeg_pipeline_path):
    print(os.listdir(eeg_pipeline_path))
else:
    print(f"Folder not found: {eeg_pipeline_path}")

Current directory: c:\Users\Ayush Batra\Documents\College\EEG\eeg-bala-sharks\sanity_checks
Parent directory: c:\Users\Ayush Batra\Documents\College\EEG\eeg-bala-sharks

Files in eeg_pipeline folder:
['behavioral_analysis.py', 'statistical_testing.py', 'time_frequency_analysis.py', '__pycache__']


In [7]:
# TEST 1: Markov Transition Matrix
# ================================
from eeg_pipeline.behavioral_analysis import compute_markov_transition_matrix, compute_predictability

print("TEST 1: Markov Transition Matrix")
print("-" * 40)

# Test with perfect cycling: R->P->S->R->P->S
choices = [0, 1, 2, 0, 1, 2]
matrix = compute_markov_transition_matrix(choices)

print(f"Input: {choices} (R->P->S->R->P->S)")
print(f"Matrix shape: {matrix.shape}")
print(f"Matrix:\n{matrix}")
print(f"Rows sum to 1: {np.allclose(matrix.sum(axis=1), 1.0)}")

# Assertions
assert matrix.shape == (3, 3), "❌ Matrix should be 3x3"
assert np.allclose(matrix.sum(axis=1), 1.0), "❌ Rows should sum to 1"
assert matrix[0, 1] == 1.0, "❌ Rock should always go to Paper in cycling"

print("✅ TEST 1 PASSED: Markov transition matrix works correctly")

TEST 1: Markov Transition Matrix
----------------------------------------
Input: [0, 1, 2, 0, 1, 2] (R->P->S->R->P->S)
Matrix shape: (3, 3)
Matrix:
[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
Rows sum to 1: True
✅ TEST 1 PASSED: Markov transition matrix works correctly


In [8]:
# TEST 2: Predictability Score
# ============================
print("\nTEST 2: Predictability Score")
print("-" * 40)

# Perfectly predictable (always repeat)
predictable_matrix = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
])
pred_high = compute_predictability(predictable_matrix)
print(f"Perfectly predictable matrix -> Score: {pred_high:.3f} (expected: 1.0)")

# Random (equal transitions)
random_matrix = np.array([
    [1/3, 1/3, 1/3],
    [1/3, 1/3, 1/3],
    [1/3, 1/3, 1/3]
])
pred_low = compute_predictability(random_matrix)
print(f"Random matrix -> Score: {pred_low:.3f} (expected: ~0.0)")

# Assertions
assert pred_high == 1.0, f"❌ Should be 1.0, got {pred_high}"
assert abs(pred_low) < 0.01, f"❌ Should be ~0, got {pred_low}"

print("✅ TEST 2 PASSED: Predictability score works correctly")


TEST 2: Predictability Score
----------------------------------------
Perfectly predictable matrix -> Score: 1.000 (expected: 1.0)
Random matrix -> Score: 0.000 (expected: ~0.0)
✅ TEST 2 PASSED: Predictability score works correctly


In [9]:
# TEST 3: Win-Stay / Lose-Shift Classification
# ============================================
from eeg_pipeline.behavioral_analysis import classify_strategy

print("\nTEST 3: Win-Stay / Lose-Shift Classification")
print("-" * 40)

# Test Win-Stay: repeat choice after every win
choices_ws = [0, 0, 0, 1, 1, 1]  # Repeat after win
outcomes_ws = [1, 1, 0, 1, 1, 0]  # Win, Win, Draw, Win, Win, Draw

result_ws = classify_strategy(choices_ws, outcomes_ws)
print(f"Win-Stay test: {result_ws['win_stay_rate']:.2%} (expected: 100%)")

# Test Lose-Shift: change choice after every loss
choices_ls = [0, 1, 2, 0, 1, 2]  # Always change
outcomes_ls = [-1, -1, -1, -1, -1, 0]  # Loss, Loss, Loss, Loss, Loss, Draw

result_ls = classify_strategy(choices_ls, outcomes_ls)
print(f"Lose-Shift test: {result_ls['lose_shift_rate']:.2%} (expected: 100%)")

# Assertions
assert result_ws['win_stay_rate'] == 1.0, f"❌ Win-Stay should be 100%, got {result_ws['win_stay_rate']:.2%}"
assert result_ls['lose_shift_rate'] == 1.0, f"❌ Lose-Shift should be 100%, got {result_ls['lose_shift_rate']:.2%}"

print("✅ TEST 3 PASSED: Strategy classification works correctly")


TEST 3: Win-Stay / Lose-Shift Classification
----------------------------------------
Win-Stay test: 100.00% (expected: 100%)
Lose-Shift test: 100.00% (expected: 100%)
✅ TEST 3 PASSED: Strategy classification works correctly


In [10]:
# TEST 4: Cycling Detection
# =========================
from eeg_pipeline.behavioral_analysis import detect_cycling

print("\nTEST 4: Cycling Detection")
print("-" * 40)

# Forward cycling: R->P->S->R->P->S
forward = [0, 1, 2, 0, 1, 2]
result_fwd = detect_cycling(forward)
print(f"Forward cycling (R→P→S): {result_fwd['forward_cycle_rate']:.2%} (expected: 100%)")

# Backward cycling: R->S->P->R->S->P
backward = [0, 2, 1, 0, 2, 1]
result_bwd = detect_cycling(backward)
print(f"Backward cycling (R→S→P): {result_bwd['backward_cycle_rate']:.2%} (expected: 100%)")

# Random expected rate
print(f"Random expected cycling rate: {result_fwd['random_expected']:.2%}")

# Assertions
assert result_fwd['forward_cycle_rate'] == 1.0, "❌ Forward cycling should be 100%"
assert result_bwd['backward_cycle_rate'] == 1.0, "❌ Backward cycling should be 100%"

print("✅ TEST 4 PASSED: Cycling detection works correctly")


TEST 4: Cycling Detection
----------------------------------------
Forward cycling (R→P→S): 100.00% (expected: 100%)
Backward cycling (R→S→P): 100.00% (expected: 100%)
Random expected cycling rate: 66.67%
✅ TEST 4 PASSED: Cycling detection works correctly


In [11]:
# TEST 5: T-tests
# ===============
from eeg_pipeline.statistical_testing import ttest_independent, ttest_paired

print("\nTEST 5: T-tests")
print("-" * 40)

np.random.seed(42)

# Independent t-test: two different groups
group1 = np.random.randn(30) + 2  # Mean ~2
group2 = np.random.randn(30)      # Mean ~0
result_ind = ttest_independent(group1, group2)

print(f"Independent t-test (different means):")
print(f"  t = {result_ind['t']:.3f}")
print(f"  p = {result_ind['p']:.4f}")
print(f"  d = {result_ind['d']:.3f} (Cohen's d)")
print(f"  Significant: {result_ind['significant']}")

# Paired t-test: same subjects, two conditions
cond1 = np.random.randn(20) + 1
cond2 = cond1 + np.random.randn(20) * 0.5 - 0.5  # Related data
result_paired = ttest_paired(cond1, cond2)

print(f"\nPaired t-test:")
print(f"  t = {result_paired['t']:.3f}")
print(f"  p = {result_paired['p']:.4f}")

# Assertions
assert 'p' in result_ind and 't' in result_ind, "❌ Missing keys in result"
assert 0 <= result_ind['p'] <= 1, "❌ p-value out of range"
assert result_ind['significant'] == True, "❌ Should detect significant difference"

print("✅ TEST 5 PASSED: T-tests work correctly")


TEST 5: T-tests
----------------------------------------
Independent t-test (different means):
  t = 8.176
  p = 0.0000
  d = 2.111 (Cohen's d)
  Significant: True

Paired t-test:
  t = 6.628
  p = 0.0000
✅ TEST 5 PASSED: T-tests work correctly


In [12]:
# TEST 6: ANOVA
# =============
from eeg_pipeline.statistical_testing import anova_oneway

print("\nTEST 6: One-way ANOVA")
print("-" * 40)

np.random.seed(42)

# Three groups with different means
rock_power = np.random.randn(50) + 0
paper_power = np.random.randn(50) + 0.5
scissors_power = np.random.randn(50) + 1.0

result = anova_oneway(rock_power, paper_power, scissors_power)

print(f"ANOVA (3 groups with different means):")
print(f"  F = {result['F']:.3f}")
print(f"  p = {result['p']:.4f}")
print(f"  η² = {result['eta_squared']:.3f} (eta-squared)")
print(f"  Significant: {result['significant']}")

# Assertions
assert 'F' in result and 'p' in result, "❌ Missing keys in result"
assert 0 <= result['p'] <= 1, "❌ p-value out of range"
assert 0 <= result['eta_squared'] <= 1, "❌ eta-squared out of range"

print("✅ TEST 6 PASSED: ANOVA works correctly")


TEST 6: One-way ANOVA
----------------------------------------
ANOVA (3 groups with different means):
  F = 20.205
  p = 0.0000
  η² = 0.216 (eta-squared)
  Significant: True
✅ TEST 6 PASSED: ANOVA works correctly


In [13]:
# TEST 7: Cohen's d Effect Size
# =============================
from eeg_pipeline.statistical_testing import cohens_d

print("\nTEST 7: Cohen's d Effect Size")
print("-" * 40)

# Large effect (d > 0.8)
large_g1 = np.array([10, 11, 12, 13, 14])
large_g2 = np.array([0, 1, 2, 3, 4])
d_large = cohens_d(large_g1, large_g2)

# Small effect (d < 0.2)
small_g1 = np.random.randn(100)
small_g2 = np.random.randn(100) + 0.1
d_small = cohens_d(small_g1, small_g2)

print(f"Large effect size: d = {d_large:.3f} (expected > 0.8)")
print(f"Small effect size: d = {d_small:.3f} (expected < 0.5)")
print(f"\nEffect size interpretation:")
print(f"  Small:  d ≈ 0.2")
print(f"  Medium: d ≈ 0.5")
print(f"  Large:  d ≈ 0.8")

# Assertions
assert d_large > 0.8, f"❌ Should be large effect, got {d_large}"

print("✅ TEST 7 PASSED: Cohen's d works correctly")


TEST 7: Cohen's d Effect Size
----------------------------------------
Large effect size: d = 6.325 (expected > 0.8)
Small effect size: d = 0.012 (expected < 0.5)

Effect size interpretation:
  Small:  d ≈ 0.2
  Medium: d ≈ 0.5
  Large:  d ≈ 0.8
✅ TEST 7 PASSED: Cohen's d works correctly


In [14]:
# TEST 8: Cluster Permutation Test (Optional)
# ===========================================
from eeg_pipeline.statistical_testing import cluster_permutation_2samp

print("\nTEST 8: Cluster Permutation Test")
print("-" * 40)
print("(Using small n_permutations for speed)")

np.random.seed(42)

# Create fake time-series data (n_trials x n_times)
n_trials = 20
n_times = 50

cond1 = np.random.randn(n_trials, n_times)
cond2 = np.random.randn(n_trials, n_times)
# Add a difference in the middle timepoints
cond2[:, 20:30] += 1.5

result = cluster_permutation_2samp(cond1, cond2, n_permutations=100)

print(f"Clusters found: {len(result['clusters'])}")
print(f"Significant clusters: {result['n_significant']}")
print(f"Cluster p-values: {result['cluster_p_values'][:3]}...")  # First 3

# Assertions
assert 't_observed' in result, "❌ Missing t_observed"
assert 'clusters' in result, "❌ Missing clusters"

print("✅ TEST 8 PASSED: Cluster permutation test works correctly")


TEST 8: Cluster Permutation Test
----------------------------------------
(Using small n_permutations for speed)
Clusters found: 4
Significant clusters: 1
Cluster p-values: [0.73 0.01 0.75]...
✅ TEST 8 PASSED: Cluster permutation test works correctly


c:\Users\Ayush Batra\Documents\College\EEG\eeg-bala-sharks\eeg_pipeline\statistical_testing.py:101: RuntimeWarning: Ignoring argument "tail", performing 1-tailed F-test
  t_obs, clusters, cluster_pv, H0 = permutation_cluster_test(


In [15]:
# TEST 9: Full Pipeline with Real Data (Optional)
# ===============================================
from eeg_pipeline.behavioral_analysis import load_behavioral_data, analyze_player
import os

print("\nTEST 9: Full Pipeline with Real Data")
print("-" * 40)

bids_root = "../MNE-sample-data/ds006761"
events_file = f"{bids_root}/sub-01/eeg/sub-01_task-RPS_events.tsv"

if os.path.exists(events_file):
    data = load_behavioral_data(events_file)
    
    print(f"Loaded {data['n_trials']} trials")
    
    # Analyze Player 1
    results = analyze_player(
        data['player1_choices'],
        data['player1_outcomes'],
        "Player 1"
    )
    
    print(f"\nPlayer 1 Results:")
    print(f"  Predictability: {results['predictability']:.3f}")
    print(f"  Win-Stay Rate: {results['win_stay_rate']:.2%}")
    print(f"  Lose-Shift Rate: {results['lose_shift_rate']:.2%}")
    print(f"  Cycling Rate: {results['cycling_rate']:.2%}")
    
    # Assertions
    assert 0 <= results['predictability'] <= 1, "❌ Predictability out of range"
    assert 0 <= results['win_stay_rate'] <= 1, "❌ Win-Stay out of range"
    
    print("✅ TEST 9 PASSED: Real data pipeline works correctly")
else:
    print("⚠️ Data file not found - skipping real data test")
    print(f"  Expected: {events_file}")


TEST 9: Full Pipeline with Real Data
----------------------------------------
Loaded 480 trials

Player 1 Results:
  Predictability: 0.020
  Win-Stay Rate: 39.38%
  Lose-Shift Rate: 55.86%
  Cycling Rate: 57.41%
✅ TEST 9 PASSED: Real data pipeline works correctly


In [16]:
# SUMMARY
# =======
print("\n" + "=" * 50)
print("SANITY CHECK SUMMARY - AYUSH'S MODULES")
print("=" * 50)

print("""
✅ TEST 1: Markov Transition Matrix
✅ TEST 2: Predictability Score  
✅ TEST 3: Win-Stay / Lose-Shift Classification
✅ TEST 4: Cycling Detection
✅ TEST 5: T-tests (Independent & Paired)
✅ TEST 6: One-way ANOVA
✅ TEST 7: Cohen's d Effect Size
✅ TEST 8: Cluster Permutation Test
✅ TEST 9: Full Pipeline with Real Data

ALL SANITY CHECKS PASSED ✓
""")

print("Modules tested:")
print("  - eeg_pipeline/behavioral_analysis.py")
print("  - eeg_pipeline/statistical_testing.py")
print("  - eeg_pipeline/time_frequency_analysis.py")


SANITY CHECK SUMMARY - AYUSH'S MODULES

✅ TEST 1: Markov Transition Matrix
✅ TEST 2: Predictability Score  
✅ TEST 3: Win-Stay / Lose-Shift Classification
✅ TEST 4: Cycling Detection
✅ TEST 5: T-tests (Independent & Paired)
✅ TEST 6: One-way ANOVA
✅ TEST 7: Cohen's d Effect Size
✅ TEST 8: Cluster Permutation Test
✅ TEST 9: Full Pipeline with Real Data

ALL SANITY CHECKS PASSED ✓

Modules tested:
  - eeg_pipeline/behavioral_analysis.py
  - eeg_pipeline/statistical_testing.py
  - eeg_pipeline/time_frequency_analysis.py
